# Flow Matching for Conditional 3D Geology — End-to-End Tutorial

This notebook walks through the **entire pipeline** for stochastic-interpolation-based flow matching,
applied to generating 3D geological models conditioned on borehole observations.

```
                   ODE Solver
  X0 (noise) ──────────────────────────────► X1 (geology)
  t=0                  Unet3D learns                 t=1
                    velocity dX/dt
                  conditioned on ATb
                  (observed boreholes)
```

## What is Stochastic Interpolation?
Stochastic interpolation defines a smooth path between a noise distribution X0 ~ N(0,I)
and a data distribution X1 ~ p_data using a time-indexed interpolant:

```
  Xt = α(t)·X0 + β(t)·X1      (one-sided linear: α(t)=1-t, β(t)=t)
  Vt = dXt/dt = α̇(t)·X0 + β̇(t)·X1  = -X0 + X1
```

A neural network (UNet) learns to predict Vt from (Xt, t, conditioning data ATb).
At inference, we integrate dX/dt = UNet(Xt, ATb, t) from t=0 → t=1 using an ODE solver.

In [1]:
import os, sys
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Add project directory to path so we can import the training/inference scripts
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, PROJECT_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. The Data

Each sample is a **3D categorical tensor** `[1, 1, 64, 64, 64]` where each voxel holds an integer rock-type category (−1 = air).
The **borehole** tensor is the same shape but with rock-type retained only along vertical drill lines — everything else is set to −1 (air/unknown).

The conditioning problem: *given only the borehole observations, generate plausible complete geology models.*

In [2]:
SAMPLES_DIR = os.path.join(PROJECT_DIR, 'samples/generative-conditional-3d/conditional_gen_demo_0')

true_model = torch.load(os.path.join(SAMPLES_DIR, 'true_model.pt'), map_location='cpu')  # [1, 1, X, Y, Z]
boreholes  = torch.load(os.path.join(SAMPLES_DIR, 'boreholes.pt'),  map_location='cpu')  # [1, 1, X, Y, Z]
solution   = torch.load(os.path.join(SAMPLES_DIR, 'sol_0.pt'),      map_location='cpu')  # [1, 1, X, Y, Z]

print('true_model shape:', true_model.shape, '| dtype:', true_model.dtype)
print('boreholes  shape:', boreholes.shape)
print('categories present:', true_model.unique().tolist())

# --- visualise a central XZ slice ---
mid = true_model.shape[-1] // 2  # midpoint in Z
cmap = plt.cm.tab20

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
titles = ['True Geology', 'Borehole Observations\n(conditioning data)', 'Generated Sample']
datas  = [true_model, boreholes, solution]

for ax, title, data in zip(axes, titles, datas):
    slc = data[0, 0, :, :, mid].numpy()   # [X, Y] slice
    im = ax.imshow(slc, cmap=cmap, vmin=-1, vmax=14, origin='lower', interpolation='nearest')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.colorbar(im, ax=axes, label='Category (−1 = air)', fraction=0.02)
plt.suptitle('Central Z-slice of 64×64×64 Geological Volume', fontsize=12)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/home/tubby/flowtrain_stochastic_interpolation/samples/generative-conditional-3d/conditional_gen_demo_0/true_model.pt'

## 2. Categorical Embedding

Flow matching operates in a **continuous embedding space**, not on discrete category integers.
Each of the 15 rock-type categories is mapped to a 15-dimensional unit vector placed at a vertex of a
centred regular simplex — maximising angular separation between categories.

```
  integer  [B, 1, X, Y, Z]  →  embed()  →  float [B, 15, X, Y, Z]
```

After generation, the reverse `decode()` maps back via nearest-neighbour cosine similarity.

In [ ]:
from model_train_sh_inference_cond import Geo3DStochInterp

CHECKPOINT = os.path.join(PROJECT_DIR, 'demo_model/conditional-weights.ckpt')
model = Geo3DStochInterp.load_from_checkpoint(CHECKPOINT, map_location=device)

# Apply EMA weights (exponential moving average = smoother, better performing weights)
if hasattr(model, 'ema_shadow') and model.ema_shadow:
    for name, param in model.net.named_parameters():
        if name in model.ema_shadow:
            param.data.copy_(model.ema_shadow[name])
    print('Applied EMA shadow weights')

model.to(device).eval()
print('Model loaded. Embedding dim:', model.embedding_dim, '| Data shape:', model.data_shape)

# Embed a single sample: integer categories → continuous vectors
X1 = model.embed(true_model.to(device))   # [1, 15, 64, 64, 64]
print('Embedded X1 shape:', X1.shape, '  (15 channels = embedding dim)')
print('Unit norm per voxel (should be ~1):', X1[:, :, 32, 32, 32].norm().item())

## 3. The Stochastic Interpolant  

The interpolant defines **how to linearly blend noise X0 into data X1** at any time t ∈ [0, 1]:

```
  Xt  = α(t)·X0 + β(t)·X1       where α(t) = 1−t,  β(t) = t
  Vt  = α̇(t)·X0 + β̇(t)·X1    = −X0 + X1
```

The UNet is trained to predict Vt given Xt and t. At t=0 the sample is pure noise; at t=1 it is data.

In [ ]:
from flowtrain.interpolation import LinearInterpolant, StochasticInterpolator

interpolant   = LinearInterpolant(one_sided=True)
interpolator  = StochasticInterpolator(interpolant)

# Demonstrate the interpolant path for a single embedding channel
X0_demo = torch.randn_like(X1)   # pure noise at t=0
X1_demo = X1.detach()

ts = torch.linspace(0, 1, 7)
fig, axes = plt.subplots(1, 7, figsize=(16, 3))

for ax, t in zip(axes, ts):
    T = t.expand(1)  # batch dim
    Xt, Vt = interpolator.flow_objective(T, X0_demo, X1_demo)
    slc = Xt[0, 0, :, :, mid].detach().cpu().numpy()  # channel 0, mid Z-slice
    ax.imshow(slc, cmap='RdBu_r', origin='lower', vmin=-2, vmax=2)
    ax.set_title(f't={t:.2f}', fontsize=9)
    ax.axis('off')

plt.suptitle('Interpolant path: Xt  (noise at t=0  →  embedded geology at t=1)', fontsize=11)
plt.tight_layout()
plt.show()

print('Vt (target velocity) is constant for linear interpolant: Vt = X1 - X0')
print('Vt norm:', Vt.norm().item())

## 4. Training Step  *(concept — no training here, just showing the logic)*

Each training step:

1. Sample a batch of geology X1 and embed it  
2. Build borehole conditioning data `ATb = embed(X1) * borehole_mask`  (zero outside known voxels)
3. Sample random time `T ~ Uniform(0, 1)`
4. Compute noisy interpolant state `Xt` and target velocity `Vt = X1 - X0`
5. Predict velocity: `Vt_hat = UNet(Xt, ATb, T)`
6. Loss = MSE(Vt, Vt_hat) + reconstruction penalty

The reconstruction penalty enforces that the model predicts borehole values correctly at the observed locations.

In [ ]:
import torch.nn.functional as F

# Build ATb (conditional data) from the saved borehole tensor
X1 = model.embed(true_model.to(device))            # [1, 15, 64, 64, 64]
bh = boreholes.to(device)
boreholes_mask = (bh != -1)                        # True where geology is observed
mask_expanded  = boreholes_mask.expand(-1, X1.shape[1], -1, -1, -1)  # [1, 15, 64, 64, 64]
ATb = X1 * mask_expanded                           # zero out unobserved voxels

print(f'ATb shape:         {ATb.shape}')
print(f'Observed voxels:   {boreholes_mask.sum().item()} / {boreholes_mask.numel()}')
print(f'Fraction observed: {boreholes_mask.float().mean().item():.2%}')

# --- Single training-step forward pass (demonstration only) ---
X0  = torch.randn_like(X1)
T   = torch.tensor([0.5], device=device)          # t = 0.5 for this demo
Xt, Vt = interpolator.flow_objective(T, X0, X1)

with torch.no_grad():
    Vt_hat = model.net(Xt, ATb, T)                # UNet prediction

mse = F.mse_loss(Vt_hat, Vt)
print(f'\nVelocity MSE at t=0.5: {mse.item():.4f}  (lower = better-trained model)')

## 5. Inference — ODE Integration

At inference, the UNet acts as the **velocity field** for an ODE:

```
  dX/dt = UNet(Xt, ATb, t)       X(0) = X0 ~ N(0, I)
```

We use `torchdiffeq.odeint` (dopri5 adaptive solver) to integrate from t=0 → t=1.
Each call to `odeint` produces a **different sample** because X0 is random.
The conditioning `ATb` stays fixed — it steers all trajectories toward geology
consistent with the observed boreholes.

In [ ]:
from flowtrain.solvers import ODEFlowSolver

# Wrap the conditional UNet: solver expects model(x, t) but we need to close over ATb
ATb_inf = ATb.expand(1, -1, -1, -1, -1)   # batch=1

def conditional_velocity(x, t):
    """dX/dt = UNet(x, ATb, t)  —  ATb is fixed conditioning from boreholes"""
    return model.net(x, ATb=ATb_inf, time=t)

solver = ODEFlowSolver(model=conditional_velocity, atol=1e-4, rtol=1e-4, method='dopri5')

# Sample initial noise and integrate t: 0 → 1
torch.manual_seed(42)
X0_inf = torch.randn(1, model.embedding_dim, *model.data_shape, device=device)  # [1, 15, 64, 64, 64]

print('Running ODE from t=0 (noise) to t=1 (geology)...')
with torch.no_grad():
    trajectory = solver.solve(X0_inf, t0=0.0001, tf=0.9999, n_steps=8)  # [T, B, 15, 64, 64, 64]

print(f'Trajectory shape: {trajectory.shape}  → [{trajectory.shape[0]} time snapshots, batch, channels, X, Y, Z]')

X1_generated = trajectory[-1]   # final state = generated geology in embedding space
print(f'Generated embedding shape: {X1_generated.shape}')

## 6. Decode & Visualise

The generated embedding `[1, 15, 64, 64, 64]` is decoded back to integer categories
by finding the nearest simplex vertex via cosine similarity.

We then compare: **true geology | borehole conditioning | generated sample**.

In [ ]:
# Decode: embedding → category indices (shifted back by -1 to restore original range)
generated_cats = model.decode(X1_generated).detach().cpu() - 1   # [1, 64, 64, 64]
print('Decoded categories shape:', generated_cats.shape)
print('Unique categories in generated sample:', generated_cats.unique().tolist())

# --- Compare three views side by side (XY, XZ, YZ mid-slices) ---
def show_3views(ax_row, vol, title, cmap=plt.cm.tab20, vmin=-1, vmax=14):
    v = vol.squeeze().numpy()   # [64, 64, 64]
    mid = v.shape[0] // 2
    slices = [v[:, :, mid], v[:, mid, :], v[mid, :, :]]
    labels = ['XY', 'XZ', 'YZ']
    for ax, slc, lbl in zip(ax_row, slices, labels):
        ax.imshow(slc, cmap=cmap, vmin=vmin, vmax=vmax, origin='lower', interpolation='nearest')
        ax.set_title(f'{title}\n({lbl} slice)', fontsize=9)
        ax.axis('off')

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
show_3views(axes[0], true_model[0],     'True Geology')
show_3views(axes[1], boreholes[0],      'Boreholes (conditioning)')
show_3views(axes[2], generated_cats[0], 'Generated Sample')

plt.suptitle('Flow Matching: Conditional 3D Geology Generation', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. ODE Trajectory Visualisation

The integration path shows how Gaussian noise gradually resolves into structured geology
as the UNet velocity field guides the ODE from t=0 → t=1.

In [ ]:
n_snaps = trajectory.shape[0]
fig, axes = plt.subplots(1, n_snaps, figsize=(3 * n_snaps, 3))

ts_shown = np.linspace(0.0001, 0.9999, n_snaps)

for i, (ax, t_val) in enumerate(zip(axes, ts_shown)):
    # Decode each time snapshot
    snap_embedding = trajectory[i]                              # [1, 15, 64, 64, 64]
    snap_cats = model.decode(snap_embedding).detach().cpu() - 1  # [1, 64, 64, 64]
    slc = snap_cats[0, :, :, mid].numpy()                      # [64, 64] XY slice
    ax.imshow(slc, cmap=plt.cm.tab20, vmin=-1, vmax=14, origin='lower', interpolation='nearest')
    ax.set_title(f't={t_val:.2f}', fontsize=9)
    ax.axis('off')

plt.suptitle('ODE Trajectory: noise (t=0) → geology (t=1)', fontsize=12)
plt.tight_layout()
plt.show()

## Summary: Full Pipeline

```
TRAINING
  X1 = embed(geology)                        # int → 15-d simplex vectors
  ATb = X1 * borehole_mask                   # zero out unobserved voxels
  X0 ~ N(0,I);   T ~ Uniform(0,1)           # random noise and time
  Xt = (1-T)*X0 + T*X1                       # interpolated state
  Vt = X1 - X0                               # target velocity (constant for linear)
  loss = MSE( UNet(Xt, ATb, T),  Vt )        # train UNet to predict velocity

INFERENCE
  ATb = embed(true_model) * borehole_mask    # build condition from observations
  X0 ~ N(0,I)                                # fresh random noise
  dX/dt = UNet(Xt, ATb, t)                   # UNet is the velocity field
  integrate t: 0 → 1  (dopri5 ODE solver)    # flow noise → geology
  output = decode(X1)                        # embedding → integer categories
```

Key files:
- **Interpolant**: `flowtrain/interpolation/interpolation.py`
- **UNet velocity field**: `flowtrain/models/unet_attn_3d_cond_v3.py`
- **ODE solver**: `flowtrain/solvers/solvers.py`
- **Lightning training module**: `project/geodata-3d-conditional/model_train_sh_inference_cond.py`